[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/S08_01_arboles_decision.ipynb)

# S08 · 01 — Árboles de decisión: sobreajuste, poda e importancia

**Tiempo estimado:** 55 min · **Sprint 8** · M3 (UD 3.6)

**Contexto TurisData.** El gerente pide algo que **pueda leer sin ser matemático**: reglas del
tipo «si la reserva es de tarifa flexible, sin depósito y con más de 100 días de antelación,
el riesgo de cancelar es alto». Los **árboles de decisión** dan exactamente eso… siempre que
no los dejemos crecer sin control.

**Al terminar sabrás:**
1. Explicar cómo un árbol parte los datos en preguntas de sí/no.
2. Detectar el **sobreajuste** de un árbol libre y controlarlo con **poda** (`max_depth`, `min_samples_leaf`, `ccp_alpha`).
3. Leer las reglas de un árbol pequeño y su **importancia de variables**.
4. Valorar sus puntos fuertes (explicable, sin escalado) y débiles (inestable).

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)

# Los árboles NO necesitan escalar; solo imputar nulos y codificar las categóricas
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()
print("Columnas tras el preprocesado:", len(nombres))

## 1. La idea: preguntas de sí/no encadenadas

Un árbol de decisión busca, entre todas las variables y umbrales posibles, **la pregunta que
mejor separa** las reservas canceladas de las no canceladas (medido con el índice de
**Gini** o la entropía: cuanto más «pura» la separación, mejor). Luego repite el proceso
en cada rama hasta que se cumple una condición de parada. Cada **hoja** final da una
predicción (la clase mayoritaria en ella).

Lo vemos con un árbol **pequeño** (profundidad 3):

In [ ]:
arbol_peq = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(Xtr, y_train)
plt.figure(figsize=(16, 6))
plot_tree(arbol_peq, feature_names=list(nombres), class_names=["se mantiene", "cancela"],
          filled=True, rounded=True, fontsize=8, proportion=True)
plt.title("Árbol de decisión de profundidad 3")
plt.show()

In [ ]:
print(export_text(arbol_peq, feature_names=list(nombres), max_depth=3))

**Ejercicio 1.** El árbol pequeño empieza preguntando por la variable que mejor separa. Guarda
en `variable_raiz` el **nombre** de esa variable (pista: `arbol_peq.tree_.feature[0]` es el índice
de la columna en `nombres`) y en `n_hojas` cuántas hojas tiene (`arbol_peq.get_n_leaves()`).

In [ ]:
# TODO: nombres[arbol_peq.tree_.feature[0]] y get_n_leaves()
...
assert isinstance(variable_raiz, str) and len(variable_raiz) > 3, "variable_raiz debe ser el nombre de una columna"
assert 2 <= n_hojas <= 8, "Un árbol de profundidad 3 tiene como máximo 8 hojas"
print("El árbol empieza preguntando por:", variable_raiz)

## 2. Sobreajuste: el árbol libre

Si no le ponemos límite, el árbol sigue preguntando hasta que **cada hoja es pura**,
es decir, **memoriza** el entrenamiento (incluido el ruido).

**Ejercicio 2.** Entrena `arbol_libre = DecisionTreeClassifier(random_state=SEMILLA)` (sin
límites) con train. Guarda `auc_train_libre` y `auc_test_libre` (AUC con `predict_proba`)
y `hojas_libre` (número de hojas).

In [ ]:
# TODO: roc_auc_score(y, modelo.predict_proba(X)[:, 1]) en train y en test
...
assert auc_train_libre > 0.99, "El árbol libre memoriza el entrenamiento: AUC ≈ 1"
assert auc_test_libre < 0.70, "…pero en test cae mucho (< 0,70): sobreajuste"
assert hojas_libre > 500, "Un árbol libre tiene cientos de hojas"
print(f"Árbol libre: {hojas_libre} hojas · AUC train {auc_train_libre:.3f} · AUC test {auc_test_libre:.3f}")

## 3. Poda: frenar el crecimiento

La **poda** limita la complejidad. Tres formas habituales:
- `max_depth`: profundidad máxima (nº de preguntas seguidas).
- `min_samples_leaf`: mínimo de reservas por hoja (evita hojas «de 2 casos»).
- `ccp_alpha`: poda **a posteriori**: se construye el árbol y se recortan las ramas que no
  compensan su complejidad (parámetro de coste-complejidad).

Miramos cómo cambia el AUC de **validación cruzada** (nunca el de test) con `min_samples_leaf`.

In [ ]:
hojas_min = [1, 5, 10, 20, 40, 80, 160, 320]
auc_cv_hojas = []
for m in hojas_min:
    arb = Pipeline([("prep", prep), ("arbol", DecisionTreeClassifier(min_samples_leaf=m, random_state=SEMILLA))])
    auc_cv_hojas.append(cross_val_score(arb, X_train, y_train, cv=cv, scoring="roc_auc").mean())
plt.figure(figsize=(7, 3.8))
plt.plot(hojas_min, auc_cv_hojas, marker="o", color="#2a6f97")
plt.xscale("log")
plt.title("Poda: AUC (validación cruzada) según el mínimo de reservas por hoja")
plt.xlabel("min_samples_leaf (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.show()

**Ejercicio 3.** Con las listas `hojas_min` y `auc_cv_hojas`, guarda en `mejor_min_hoja` el valor
con mejor AUC. Entrena `arbol_podado` con ese valor sobre train y guarda `auc_test_podado`
(AUC en test).

In [ ]:
# TODO: np.argmax(auc_cv_hojas) da la posición del mejor; después DecisionTreeClassifier(min_samples_leaf=...)
...
assert mejor_min_hoja >= 10, "Con estos datos ruidosos, hojas de al menos 10 reservas funcionan mejor"
assert auc_test_podado > auc_test_libre + 0.08, "El árbol podado debería ganar claramente al libre en test"
print(f"min_samples_leaf = {mejor_min_hoja} · AUC test: {auc_test_podado:.3f} (libre: {auc_test_libre:.3f}) · hojas: {arbol_podado.get_n_leaves()}")

## 4. Importancia de variables

El árbol calcula cuánto **reduce la impureza** (Gini) cada variable en total: es la
`feature_importances_` (suma 1). Ayuda a explicar *qué mira* el modelo. Ojo:
- es una importancia **del modelo**, no una relación causal;
- tiende a inflar las variables numéricas con muchos valores distintos;
- en variables correlacionadas se reparte de forma algo arbitraria.

In [ ]:
importancias = pd.Series(arbol_podado.feature_importances_, index=nombres).sort_values(ascending=False)
plt.figure(figsize=(7, 4.5))
importancias.head(10).iloc[::-1].plot(kind="barh", color="#2a6f97")
plt.title("Importancia de variables del árbol podado (top 10)")
plt.xlabel("Importancia (reducción de impureza)")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Ejercicio 4.** Guarda en `top3` la lista de los **nombres** de las 3 variables más importantes de
`arbol_podado` (usa la Series `importancias` ya ordenada) y en `suma_top3` cuánta importancia
acumulan entre las tres.

In [ ]:
# TODO: importancias.head(3).index y .sum()
...
assert len(top3) == 3 and all(isinstance(n, str) for n in top3), "top3 debe ser una lista de 3 nombres"
assert 0.3 < suma_top3 <= 1, "Las tres primeras deberían acumular una parte importante"
print("Top 3:", top3, f"· acumulan {suma_top3:.0%} de la importancia")

## 5. Inestabilidad: el gran defecto

Un árbol cambia si cambian un poco los datos. Lo comprobamos entrenando 5 árboles (iguales
en configuración) con distintas submuestras del 80 % de train y comparando **qué predicen para
las mismas reservas de test**.

In [ ]:
predicciones = []
for semilla in range(5):
    idx = np.random.default_rng(semilla).choice(len(Xtr), int(0.8 * len(Xtr)), replace=False)
    a = DecisionTreeClassifier(min_samples_leaf=10, random_state=SEMILLA).fit(Xtr[idx], y_train.to_numpy()[idx])
    predicciones.append(a.predict(Xte))
predicciones = np.array(predicciones)
# % de reservas de test en las que los 5 árboles NO se ponen de acuerdo
desacuerdo = (predicciones.min(axis=0) != predicciones.max(axis=0)).mean()
print(f"En el {desacuerdo:.0%} de las reservas de test, los 5 árboles no coinciden en la predicción.")

Este defecto es el que arreglan los *ensembles* (bosques aleatorios y boosting) que verás a continuación.

## 6. Mini-reto: reglas para el gerente

Entrena `arbol_gerente = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEMILLA)`
con train. Después escribe en la lista `reglas` **tres frases** en lenguaje de negocio (con cifras
si puedes) que resuman lo que hace el árbol (mira `export_text`). Guarda su AUC en test en
`auc_test_gerente`.

In [ ]:
# TODO: entrena el árbol, imprime export_text y redacta las reglas a partir de lo que veas
...
assert 0.6 < auc_test_gerente < 0.85, "El AUC del árbol de profundidad 3 debería estar entre 0,6 y 0,85"
assert len(reglas) == 3 and all(isinstance(r, str) and len(r) > 20 for r in reglas), "Escribe tres frases completas"

## 7. Para reflexionar

1. El árbol libre acierta el 100 % en train. ¿Por qué es una mala noticia?
2. ¿Qué pierdes al limitar la profundidad y qué ganas?
3. ¿Por qué los árboles no necesitan escalar las variables?
4. Si el gerente pide «por qué se marcó esta reserva concreta como de riesgo», ¿qué ventaja tiene un árbol frente a una red neuronal?

## 8. Resumen: qué has aprendido
- Un árbol = preguntas de sí/no encadenadas que maximizan la pureza de cada rama.
- Sin límites **memoriza**; la poda (`max_depth`, `min_samples_leaf`, `ccp_alpha`) controla el sobreajuste.
- Es muy **explicable** y no necesita escalado, pero es **inestable** y suele rendir menos que los ensembles.